# 02 — Tune GBDT Models

Optuna study comparing XGBoost, LightGBM, and CatBoost
on static features. Best model logged to MLflow.

In [ ]:
input_dir = "data/processed"
n_trials = 100
random_state = 42
cv_folds = 5

In [ ]:
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import cross_val_score, StratifiedKFold
import mlflow

In [ ]:
# ── Load data (no scaling needed for GBDT) ──
X_train = pd.read_parquet(f"{input_dir}/X_train.parquet")
y_train = pd.read_parquet(f"{input_dir}/y_train.parquet")["y"]
print(f"Loaded {len(X_train)} training rows, {X_train.shape[1]} features")

In [ ]:
cv = StratifiedKFold(cv_folds, shuffle=True, random_state=random_state)


def objective(trial):
    model_type = trial.suggest_categorical("model", ["xgb", "lgbm", "catboost"])

    if model_type == "xgb":
        import xgboost as xgb

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 2000, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        }
        model = xgb.XGBClassifier(**params, eval_metric="logloss", random_state=random_state)
    elif model_type == "lgbm":
        import lightgbm as lgb

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 2000, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 16, 256, log=True),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.3, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        }
        model = lgb.LGBMClassifier(**params, random_state=random_state, verbose=-1)
    else:
        from catboost import CatBoostClassifier

        params = {
            "iterations": trial.suggest_int("iterations", 100, 2000, log=True),
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-8, 10.0, log=True),
        }
        model = CatBoostClassifier(**params, verbose=0, random_seed=random_state)

    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc")
    return scores.mean()


study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=n_trials)
print(f"Best trial: {study.best_trial.value:.4f} ROC-AUC")
print(f"Best params: {study.best_trial.params}")

In [ ]:
# ── Log best model to MLflow ──
mlflow.set_experiment("gbdt_models")
with mlflow.start_run():
    mlflow.log_params(study.best_trial.params)
    mlflow.log_metric("best_cv_roc_auc", study.best_trial.value)
    mlflow.log_metric("n_trials", n_trials)

    # Train best model on full training set
    best_params = study.best_trial.params
    model_type = best_params.pop("model")
    if model_type == "xgb":
        import xgboost as xgb

        model = xgb.XGBClassifier(**best_params, eval_metric="logloss", random_state=random_state)
    elif model_type == "lgbm":
        import lightgbm as lgb

        model = lgb.LGBMClassifier(**best_params, random_state=random_state, verbose=-1)
    else:
        from catboost import CatBoostClassifier

        model = CatBoostClassifier(**best_params, verbose=0, random_seed=random_state)

    model.fit(X_train, y_train)
    mlflow.sklearn.log_model(model, "gbdt_model")
    print(f"Best GBDT model: {model_type}")
    print(f"Logged to MLflow run: {mlflow.active_run().info.run_id}")